# HistGradientBoosting Hyperparameter Tuning — `SEG_A` vs `SEG_B/C`

This notebook tunes the binary Stage 1 classifier used in the hierarchical segmentation strategy.

**Objective:** identify clear `SEG_A` HCPs and route potential `SEG_B/C` HCPs into the second-stage model.

**Main steps**
1. Load temporal tensor features and labels.
2. Keep only labeled HCPs and convert the target to binary.
3. Split data by predefined HCP folds.
4. Run a randomized hyperparameter search for `HistGradientBoostingClassifier`.
5. Tune the probability decision threshold.
6. Retrain the selected model on train + validation data.
7. Evaluate on the test fold and save model artifacts.

## 1. Imports and Reproducibility

Load required libraries and fix the random seed so experiments are more reproducible.

In [ ]:
import os
import json
import joblib
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Paths and Output Folders

Define local folders for tensors, model artifacts, and prediction outputs.

In [ ]:
TENSOR_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_tensors")
MODEL_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_models")
OUTPUT_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_outputs")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load Tensor Data

Load the 3D feature tensor, encoded labels, and predefined HCP fold assignments.

In [ ]:
X_torch = torch.load(TENSOR_DIR / "X_features.pt")
y_torch = torch.load(TENSOR_DIR / "y_labels.pt")
fold_torch = torch.load(TENSOR_DIR / "folds.pt")

X_tf = tf.convert_to_tensor(X_torch.cpu().numpy(), dtype=tf.float32)
y_tf = tf.convert_to_tensor(y_torch.cpu().numpy(), dtype=tf.int64)
fold_tf = tf.convert_to_tensor(fold_torch.cpu().numpy(), dtype=tf.int64)

print("X shape:", X_tf.shape)
print("y shape:", y_tf.shape)
print("folds shape:", fold_tf.shape)

C:\Users\omarl\AppData\Local\Temp\ipykernel_28040\3665750501.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_torch = torch.load(TENSOR_DIR / "X_features.pt")
C:\Users\

X shape: (20931, 86, 65)
y shape: (20931,)
folds shape: (20931,)


## 4. Load Tensor Manifest

Load the manifest aligned with the tensor rows. This is useful to reconnect predictions to HCP identifiers later.

In [ ]:
manifest_path = TENSOR_DIR / "tensor_manifest.csv"

if manifest_path.exists():
    manifest_df = pd.read_csv(manifest_path)
    print("Manifest loaded:", manifest_df.shape)
    display(manifest_df.head())
else:
    manifest_df = None
    print("No manifest found.")

Manifest loaded: (20931, 3)


,NUEVO_ID,ATSEG_HCP,HCP_FOLD
0,1,NaN,-1
1,2,NaN,-1
2,3,SEG_A,1
3,4,NaN,-1
4,5,NaN,-1


### Manifest Alignment Check

Validate that the manifest has the same number of rows as the feature tensor.

In [ ]:
if manifest_df is not None:
    assert len(manifest_df) == X_tf.shape[0], "Manifest and tensor rows are not aligned."

## 5. Keep Only Labeled HCPs

Filter out unlabeled HCPs (`label = -1`) so the model is trained and evaluated only on known segments.

In [ ]:
labeled_mask = y_tf != -1

X_labeled = tf.boolean_mask(X_tf, labeled_mask)
y_labeled = tf.boolean_mask(y_tf, labeled_mask)
fold_labeled = tf.boolean_mask(fold_tf, labeled_mask)

print("X labeled:", X_labeled.shape)
print("y labeled:", y_labeled.shape)
print("fold labeled:", fold_labeled.shape)

print(pd.Series(y_labeled.numpy()).value_counts().sort_index())

X labeled: (11899, 86, 65)
y labeled: (11899,)
fold labeled: (11899,)
0    6406
1    3349
2    2144
Name: count, dtype: int64


## 6. Binary Target Definition

Convert the original 3-class target into a binary target: `0 = SEG_A`, `1 = SEG_B/C`.

In [ ]:
y_binary = tf.cast(y_labeled != 0, tf.int64)

binary_target_names = ["SEG_A", "SEG_BC"]

print(pd.Series(y_binary.numpy()).value_counts().sort_index())

0    6406
1    5493
Name: count, dtype: int64


## 7. Train / Validation / Test Split by Fold

Use predefined HCP folds to avoid leakage between train, validation, and test sets.

In [ ]:
test_fold = 3
val_fold = 4

train_mask = (fold_labeled != test_fold) & (fold_labeled != val_fold)
val_mask = fold_labeled == val_fold
test_mask = fold_labeled == test_fold

X_train_tensor = tf.boolean_mask(X_labeled, train_mask)
y_train_tensor = tf.boolean_mask(y_binary, train_mask)

X_val_tensor = tf.boolean_mask(X_labeled, val_mask)
y_val_tensor = tf.boolean_mask(y_binary, val_mask)

X_test_tensor = tf.boolean_mask(X_labeled, test_mask)
y_test_tensor = tf.boolean_mask(y_binary, test_mask)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Val:", X_val_tensor.shape, y_val_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: (7140, 86, 65) (7140,)
Val: (2379, 86, 65) (2379,)
Test: (2380, 86, 65) (2380,)


## 8. Flatten Tensor Features

Tree-based models require a 2D matrix, so each HCP's `(weeks × features)` tensor is flattened into one vector.

In [ ]:
X_train_flat = X_train_tensor.numpy().reshape(X_train_tensor.shape[0], -1)
X_val_flat = X_val_tensor.numpy().reshape(X_val_tensor.shape[0], -1)
X_test_flat = X_test_tensor.numpy().reshape(X_test_tensor.shape[0], -1)

y_train_bin = y_train_tensor.numpy()
y_val_bin = y_val_tensor.numpy()
y_test_bin = y_test_tensor.numpy()

print("X_train_flat:", X_train_flat.shape)
print("X_val_flat:", X_val_flat.shape)
print("X_test_flat:", X_test_flat.shape)

X_train_flat: (7140, 5590)
X_val_flat: (2379, 5590)
X_test_flat: (2380, 5590)


## 9. Class Distribution Checks

Review class balance across train, validation, and test splits.

In [ ]:
def show_binary_distribution(y, name):
    counts = pd.Series(y).value_counts().sort_index()
    pct = pd.Series(y).value_counts(normalize=True).sort_index() * 100
    
    df_dist = pd.DataFrame({
        "class_encoded": counts.index,
        "class_label": [binary_target_names[i] for i in counts.index],
        "count": counts.values,
        "pct": pct.round(2).values
    })
    
    print(name)
    display(df_dist)

show_binary_distribution(y_train_bin, "Train distribution")
show_binary_distribution(y_val_bin, "Validation distribution")
show_binary_distribution(y_test_bin, "Test distribution")

Train distribution


,class_encoded,class_label,count,pct
0,0,SEG_A,3844,53.84
1,1,SEG_BC,3296,46.16


Validation distribution


,class_encoded,class_label,count,pct
0,0,SEG_A,1281,53.85
1,1,SEG_BC,1098,46.15


Test distribution


,class_encoded,class_label,count,pct
0,0,SEG_A,1281,53.82
1,1,SEG_BC,1099,46.18


## 10. Evaluation and Threshold Functions

Define reusable functions to compute metrics and select the best probability threshold.

In [ ]:
def evaluate_binary_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)

    report = classification_report(
        y_true,
        y_pred,
        target_names=["SEG_A", "SEG_BC"],
        output_dict=True,
        zero_division=0
    )

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "SEG_A_precision": report["SEG_A"]["precision"],
        "SEG_A_recall": report["SEG_A"]["recall"],
        "SEG_A_f1": report["SEG_A"]["f1-score"],
        "SEG_BC_precision": report["SEG_BC"]["precision"],
        "SEG_BC_recall": report["SEG_BC"]["recall"],
        "SEG_BC_f1": report["SEG_BC"]["f1-score"],
    }


def find_best_binary_threshold(y_true, y_proba):
    rows = []

    for threshold in np.arange(0.30, 0.71, 0.05):
        threshold = round(float(threshold), 2)

        metrics = evaluate_binary_threshold(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold
        )

        metrics["custom_score"] = (
            0.45 * metrics["macro_f1"] +
            0.35 * metrics["roc_auc"] +
            0.20 * metrics["SEG_BC_recall"]
        )

        rows.append(metrics)

    return pd.DataFrame(rows).sort_values(
        ["custom_score", "macro_f1", "roc_auc", "SEG_BC_recall"],
        ascending=False
    )

## 11. Hyperparameter Search Space

Define candidate values for the randomized search.

In [ ]:
hgb_param_grid = {
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "max_iter": [150, 250, 400, 600],
    "max_leaf_nodes": [15, 31, 63],
    "max_depth": [None, 3, 5, 7],
    "min_samples_leaf": [10, 20, 40, 60],
    "l2_regularization": [0.0, 0.01, 0.1, 1.0, 5.0],
    "max_bins": [64, 128, 255],
}

n_iter = 100

hgb_param_candidates = list(ParameterSampler(
    hgb_param_grid,
    n_iter=n_iter,
    random_state=SEED
))

len(hgb_param_candidates)

100

## 12. Randomized Hyperparameter Search

Train multiple HGB candidates on the training folds, select thresholds on validation, and rank candidates with a custom score.

In [ ]:
hgb_results = []

sample_weights_bin = compute_sample_weight(
    class_weight="balanced",
    y=y_train_bin
)

for i, params in enumerate(hgb_param_candidates, start=1):
    print("\n" + "=" * 80)
    print(f"Training HGB candidate {i}/{len(hgb_param_candidates)}")
    print(params)
    print("=" * 80)

    model = HistGradientBoostingClassifier(
        **params,
        random_state=SEED,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20
    )

    model.fit(
        X_train_flat,
        y_train_bin,
        sample_weight=sample_weights_bin
    )

    val_proba = model.predict_proba(X_val_flat)[:, 1]

    threshold_df = find_best_binary_threshold(
        y_true=y_val_bin,
        y_proba=val_proba
    )

    best_threshold_row = threshold_df.iloc[0].to_dict()

    result = {
        "candidate": i,
        **params,
        **best_threshold_row
    }

    hgb_results.append(result)

    print(
        f"Best threshold: {best_threshold_row['threshold']} | "
        f"Macro F1: {best_threshold_row['macro_f1']:.4f} | "
        f"ROC AUC: {best_threshold_row['roc_auc']:.4f} | "
        f"SEG_BC recall: {best_threshold_row['SEG_BC_recall']:.4f} | "
        f"Custom score: {best_threshold_row['custom_score']:.4f}"
    )

hgb_results_df = pd.DataFrame(hgb_results)

hgb_results_df.sort_values(
    ["custom_score", "macro_f1", "roc_auc", "SEG_BC_recall"],
    ascending=False
).head(15)


Training HGB candidate 1/100
{'min_samples_leaf': 40, 'max_leaf_nodes': 63, 'max_iter': 250, 'max_depth': 7, 'max_bins': 128, 'learning_rate': 0.05, 'l2_regularization': 0.1}
Best threshold: 0.35 | Macro F1: 0.7482 | ROC AUC: 0.8421 | SEG_BC recall: 0.8106 | Custom score: 0.7935

Training HGB candidate 2/100
{'min_samples_leaf': 10, 'max_leaf_nodes': 63, 'max_iter': 600, 'max_depth': 3, 'max_bins': 128, 'learning_rate': 0.03, 'l2_regularization': 0.0}
Best threshold: 0.35 | Macro F1: 0.7443 | ROC AUC: 0.8421 | SEG_BC recall: 0.8279 | Custom score: 0.7953

Training HGB candidate 3/100
{'min_samples_leaf': 40, 'max_leaf_nodes': 15, 'max_iter': 250, 'max_depth': None, 'max_bins': 128, 'learning_rate': 0.1, 'l2_regularization': 0.01}
Best threshold: 0.35 | Macro F1: 0.7495 | ROC AUC: 0.8429 | SEG_BC recall: 0.8206 | Custom score: 0.7964

Training HGB candidate 4/100
{'min_samples_leaf': 40, 'max_leaf_nodes': 15, 'max_iter': 400, 'max_depth': 7, 'max_bins': 64, 'learning_rate': 0.07, 'l2_r

,candidate,min_samples_leaf,max_leaf_nodes,max_iter,max_depth,max_bins,learning_rate,l2_regularization,threshold,accuracy,macro_f1,weighted_f1,roc_auc,SEG_A_precision,SEG_A_recall,SEG_A_f1,SEG_BC_precision,SEG_BC_recall,SEG_BC_f1,custom_score
69,70,60,31,600,7.0,64,0.05,0.10,0.35,0.756200,0.756200,0.756176,0.846161,0.820091,0.701015,0.755892,0.701713,0.820583,0.756507,0.800563
97,98,60,63,250,3.0,64,0.05,5.00,0.35,0.751997,0.751937,0.751640,0.843493,0.825636,0.683841,0.748079,0.692716,0.831512,0.755795,0.799896
25,26,10,15,600,3.0,64,0.03,0.10,0.35,0.751156,0.751089,0.750775,0.843980,0.825307,0.682279,0.747009,0.691667,0.831512,0.755170,0.799685
30,31,10,15,250,3.0,64,0.03,0.10,0.35,0.751156,0.751089,0.750775,0.843980,0.825307,0.682279,0.747009,0.691667,0.831512,0.755170,0.799685
82,83,20,15,600,5.0,64,0.01,5.00,0.35,0.752837,0.752818,0.752650,0.844571,0.821727,0.690867,0.750636,0.695853,0.825137,0.755000,0.799395
63,64,20,15,400,NaN,64,0.07,0.10,0.35,0.754098,0.754096,0.754032,0.845451,0.819266,0.697112,0.753269,0.698991,0.820583,0.754922,0.799368
45,46,60,15,400,7.0,64,0.07,0.00,0.40,0.768810,0.768409,0.769151,0.844976,0.805347,0.752537,0.778047,0.731810,0.787796,0.758772,0.799085
62,63,10,15,400,3.0,64,0.10,1.00,0.35,0.750315,0.750283,0.750065,0.845029,0.820728,0.686183,0.747449,0.692661,0.825137,0.753117,0.798415
40,41,40,15,250,NaN,128,0.05,1.00,0.35,0.751997,0.751987,0.751866,0.844457,0.819021,0.692428,0.750423,0.695988,0.821494,0.753551,0.798253
20,21,60,15,400,7.0,255,0.07,5.00,0.35,0.751997,0.751991,0.751903,0.844671,0.817847,0.693989,0.750845,0.696594,0.819672,0.753138,0.797965


## 13. Save Search Results

Export the hyperparameter search results for auditability and comparison.

In [ ]:
hgb_results_path = MODEL_DIR / "hgb_binary_hyperparameter_search_results.csv"

hgb_results_df.to_csv(hgb_results_path, index=False)

print("Saved:", hgb_results_path)

## 14. Select Best Candidate

Select the candidate with the strongest validation custom score.

In [ ]:
best_hgb_row = hgb_results_df.sort_values(
    ["custom_score", "macro_f1", "roc_auc", "SEG_BC_recall"],
    ascending=False
).iloc[0]

best_hgb_row

candidate             70.000000
min_samples_leaf      60.000000
max_leaf_nodes        31.000000
max_iter             600.000000
max_depth              7.000000
max_bins              64.000000
learning_rate          0.050000
l2_regularization      0.100000
threshold              0.350000
accuracy               0.756200
macro_f1               0.756200
weighted_f1            0.756176
roc_auc                0.846161
SEG_A_precision        0.820091
SEG_A_recall           0.701015
SEG_A_f1               0.755892
SEG_BC_precision       0.701713
SEG_BC_recall          0.820583
SEG_BC_f1              0.756507
custom_score           0.800563
Name: 69, dtype: float64

### Extract Final Hyperparameters

Convert the selected row into a clean parameter dictionary for the final model.

In [ ]:
hgb_param_cols = [
    "learning_rate",
    "max_iter",
    "max_leaf_nodes",
    "max_depth",
    "min_samples_leaf",
    "l2_regularization",
    "max_bins"
]

best_hgb_params = {
    col: best_hgb_row[col]
    for col in hgb_param_cols
}

best_hgb_params["max_iter"] = int(best_hgb_params["max_iter"])
best_hgb_params["max_leaf_nodes"] = int(best_hgb_params["max_leaf_nodes"])
best_hgb_params["min_samples_leaf"] = int(best_hgb_params["min_samples_leaf"])
best_hgb_params["max_bins"] = int(best_hgb_params["max_bins"])

if pd.isna(best_hgb_params["max_depth"]):
    best_hgb_params["max_depth"] = None
else:
    best_hgb_params["max_depth"] = int(best_hgb_params["max_depth"])

best_hgb_threshold = float(best_hgb_row["threshold"])

print("Best HGB params:")
print(best_hgb_params)
print("Best threshold:", best_hgb_threshold)

Best HGB params:
{'learning_rate': 0.05, 'max_iter': 600, 'max_leaf_nodes': 31, 'max_depth': 7, 'min_samples_leaf': 60, 'l2_regularization': 0.1, 'max_bins': 64}
Best threshold: 0.35


## 15. Retrain Final HGB Model

Retrain the selected configuration on train + validation data before the final test evaluation.

In [ ]:
X_trainval_flat = np.vstack([X_train_flat, X_val_flat])
y_trainval_bin = np.concatenate([y_train_bin, y_val_bin])

sample_weights_trainval_bin = compute_sample_weight(
    class_weight="balanced",
    y=y_trainval_bin
)

best_hgb_model = HistGradientBoostingClassifier(
    **best_hgb_params,
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)

best_hgb_model.fit(
    X_trainval_flat,
    y_trainval_bin,
    sample_weight=sample_weights_trainval_bin
)

HistGradientBoostingClassifier(early_stopping=True, l2_regularization=0.1,
                               learning_rate=0.05, max_bins=64, max_depth=7,
                               max_iter=600, min_samples_leaf=60,
                               n_iter_no_change=20, random_state=42)

## 16. Final Test Evaluation

Evaluate the selected model on the held-out test fold using the selected threshold.

In [ ]:
test_hgb_proba = best_hgb_model.predict_proba(X_test_flat)[:, 1]

test_hgb_metrics = evaluate_binary_threshold(
    y_true=y_test_bin,
    y_proba=test_hgb_proba,
    threshold=best_hgb_threshold
)

test_hgb_metrics

{'threshold': 0.35,
 'accuracy': 0.7487394957983193,
 'macro_f1': 0.748713943088627,
 'weighted_f1': 0.7485201683734606,
 'roc_auc': 0.8488008756807516,
 'SEG_A_precision': 0.8176744186046512,
 'SEG_A_recall': 0.6861826697892272,
 'SEG_A_f1': 0.7461799660441426,
 'SEG_BC_precision': 0.6919540229885057,
 'SEG_BC_recall': 0.821656050955414,
 'SEG_BC_f1': 0.7512479201331115}

### Classification Report

Display precision, recall, and F1-score for each binary class.

In [ ]:
test_hgb_pred = (test_hgb_proba >= best_hgb_threshold).astype(int)

print(classification_report(
    y_test_bin,
    test_hgb_pred,
    target_names=["SEG_A", "SEG_BC"]
))

print("ROC AUC:", roc_auc_score(y_test_bin, test_hgb_proba))

              precision    recall  f1-score   support

       SEG_A       0.80      0.75      0.77      1281
      SEG_BC       0.73      0.78      0.76      1099

    accuracy                           0.77      2380
   macro avg       0.76      0.77      0.76      2380
weighted avg       0.77      0.77      0.77      2380

ROC AUC: 0.8482255176269109


## Final Notes

The selected HistGradientBoosting model is used as the **Stage 1 router** in the hierarchical strategy.

The most important output of this notebook is the saved `.joblib` model and metadata, including the selected **decision threshold**. The threshold is part of the final decision rule:

`if P(SEG_B/C) >= threshold → SEG_B/C`, otherwise `SEG_A`.

The previous final model did not improve after hyperparameter tuning, but this process ensures we have a systematic approach to model selection and can be revisited in the future with different search spaces or additional data.